# Trend Lifespan & Viral Prediction AI (Mamba Edition)

This notebook contains the complete pipeline for analyzing Reddit trends, predicting their viral lifespan using a **Pure PyTorch Mamba (State Space Model)**, and generating AI summaries.

**Instructions:**
1. Run the **Setup & Imports** cell first.
2. Run the **Mamba Model & Utils** cell.
3. Run the **Execution** cell.

In [50]:
# --- CELL 1: SETUP & IMPORTS ---
import requests
import pandas as pd
import time
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from IPython.display import display, Markdown
import warnings

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")

Imports loaded successfully.


In [51]:
# --- CELL 2: CORE FUNCTIONS & MODELS (MAMBA ENABLED) ---

# 1. Minimal Mamba Layer (Pure PyTorch Implementation)
class MinimalMambaLayer(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.dt_rank = object = int(d_model / 16)
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )

        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        self.A_log = nn.Parameter(torch.log(torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        batch, seq_len, _ = x.shape
        x_and_res = self.in_proj(x)  # [batch, seq, 2*inner]
        (x, res) = x_and_res.split(split_size=[self.d_inner, self.d_inner], dim=-1)

        x = x.transpose(1, 2)
        x = self.conv1d(x)[:, :, :seq_len]
        x = x.transpose(1, 2)

        x = F.silu(x)
        
        # Simple SSM Scan (Sequential for compatibility)
        y = self.ssm_scan(x)
        
        y = y * F.silu(res)
        output = self.out_proj(y)
        return output

    def ssm_scan(self, x):
        # Simplified Selective Scan
        batch, seq_len, d_inner = x.shape
        delta_BC = self.x_proj(x) # [batch, seq, dt_rank + 2*state]
        
        delta, B, C = torch.split(delta_BC, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta)) # [batch, seq, inner]
        
        A = -torch.exp(self.A_log) # [inner, state]
        
        # Input-dependent B, C
        # For demo purposes, we will treat this as a simple gated recurrence scan
        # to avoid complex C++/CUDA requirements on Windows.
        
        ys = []
        h = torch.zeros(batch, d_inner, self.d_state, device=x.device)
        
        for t in range(seq_len):
            dt = delta[:, t, :].unsqueeze(-1) # [batch, inner, 1]
            dA = torch.exp(A * dt) # [batch, inner, state]
            dB = (dt * B[:, t, :].unsqueeze(1))
            
            xt = x[:, t, :].unsqueeze(-1) # [batch, inner, 1]
            
            h = h * dA + dB * xt
            y_t = torch.sum(h * C[:, t, :].unsqueeze(1), dim=-1) # [batch, inner]
            ys.append(y_t)
            
        y = torch.stack(ys, dim=1)
        return y + x * self.D

# 2. Temporal Model using Mamba
class TemporalSSM(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # Using our Pure PyTorch Mamba definition
        self.mamba = MinimalMambaLayer(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        out = self.mamba(x)
        return self.norm(out)

class LifespanPredictor(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.ReLU()
        )
    def forward(self, x):
        return self.model(x)

# 3. Reddit Scraper
def fetch_reddit_posts(keyword, pages=10):
    headers = {"User-Agent": "trendscopeAI/1.0 (Academic Research Project)"}
    all_posts = []
    after = None
    for i in range(pages):
        url = f"https://www.reddit.com/search.json?q={keyword}&sort=top&t=all"
        if after: url += f"&after={after}"
        try:
            r = requests.get(url, headers=headers)
            if r.status_code != 200: break
            data = r.json()
            children = data["data"]["children"]
            if not children: break
            for post in children:
                p = post["data"]
                all_posts.append({
                    "post_id": p.get("id"),
                    "text": p.get("title", ""),
                    "upvotes": p.get("score", 0),
                    "comments": p.get("num_comments", 0),
                    "created_utc": p.get("created_utc"),
                    "platform": "Reddit",
                    "keyword": keyword
                })
            after = data["data"].get("after")
            if not after: break
            time.sleep(1)
        except Exception as e:
            print(f"Error fetching data: {e}")
            break
    return pd.DataFrame(all_posts)

# 4. Semantic Filtering
def filter_by_keyword_relevance(df, keyword, embedder, threshold=0.35):
    keyword_emb = embedder.encode(keyword).reshape(1, -1)
    similarities = []
    for emb in df["embedding"]:
        sim = cosine_similarity(emb.reshape(1, -1), keyword_emb)[0][0]
        similarities.append(sim)
    df = df.copy()
    df["similarity"] = similarities
    return df[df["similarity"] >= threshold]

# 5. Data Aggregation
def aggregate_daily(df, embedder):
    if df.empty: return None, None
    df["date"] = df["created_utc"].apply(lambda x: datetime.datetime.fromtimestamp(x).date())
    df = df[df["text"].str.len() > 10]
    if "embedding" not in df.columns:
         df["embedding"] = df["text"].apply(lambda x: embedder.encode(x))
    
    daily_vectors = []
    daily_meta = []
    for date, g in df.groupby("date"):
        text_emb = np.mean(np.vstack(g["embedding"]), axis=0)
        engagement = np.array([g["upvotes"].mean(), g["comments"].mean()])
        vector = np.concatenate([text_emb, engagement])
        daily_vectors.append(vector)
        daily_meta.append({"date": date, "posts": len(g)})
    if not daily_vectors: return None, None
    X = np.vstack(daily_vectors)
    return X, daily_meta

# 6. Utilities
def summarize_trend(df):
    top = df.sort_values("upvotes", ascending=False).head(5)
    bullets = []
    for t in top["text"]:
        bullets.append("- " + t[:120] + ("..." if len(t) > 120 else ""))
    return "\n".join(bullets)

def format_reddit_link(post_id):
    return f"https://redd.it/{post_id}"

def get_trending_posts_display(df):
    top = df.sort_values("upvotes", ascending=False).head(5)
    links = []
    for i, row in top.iterrows():
        title = row['text'][:100].replace("[", "(").replace("]", ")") + ("..." if len(row['text']) > 100 else "")
        url = format_reddit_link(row['post_id'])
        links.append(f"- [{title}]({url})")
    return "\n".join(links)

def predict_viral_days(X, encoder, head):
    x_seq = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        h = encoder(x_seq)
        # Mamba returns [batch, seq, dim], take last state
        final_state = h[:, -1, :]
        pred = head(x_seq)[:, -1, :]
    return max(1, int(pred.item()))

def get_llm_summary(df):
    top_text = " ".join(df.sort_values("upvotes", ascending=False).head(10)["text"].tolist())
    # FIX: Use 'Falconsai/text_summarization' (T5-small) which uses safetensors by default
    # to bypass torch.load vulnerability on older torch versions.
    try:
        summarizer = pipeline("summarization", model="Falconsai/text_summarization", framework="pt")
        # Prefix with 'summarize: ' for T5 models
        input_text = "summarize: " + top_text
        summary = summarizer(input_text, max_length=100, min_length=30, do_sample=False)[0]['summary_text']
    except Exception as e:
        # Fallback to simple extraction if model fails (to prevent crash)
        print(f"Warning: AI Summary failed ({e}). Using simple extraction.")
        summary = top_text[:300] + "... (AI Summary unavailable due to model loading error)"
    return summary

print("Mamba Model & Core Functions initialized.")

Mamba Model & Core Functions initialized.


In [52]:
# --- CELL 3: MAIN PIPELINE ---

def run_advanced_trend_demo(keyword):
    print(f"\nAnalyzing '{keyword}'... (This may take a moment)")
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    
    print("Fetching Reddit posts...")
    df = fetch_reddit_posts(keyword, pages=10)
    if df.empty: return print("No data found.")
    
    print("Processing embeddings (using SentenceTransformer)... Mamba Model downstream.")
    df['embedding'] = df['text'].apply(lambda x: embedder.encode(x))
    df = filter_by_keyword_relevance(df, keyword, embedder)
    if df.empty: return print("No relevant data after filtering.")
    
    X, meta = aggregate_daily(df, embedder)
    if X is None or len(X) < 1:
        past_days = 0
        future_days = 0
    else:
        past_days = len(meta)
        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Initialize Mamba-based Temporal Model
        print("Initializing Mamba Temporal Model...")
        encoder = TemporalSSM(X.shape[-1])
        head = LifespanPredictor(X.shape[-1])
        future_days = predict_viral_days(X_scaled, encoder, head)

    print("Generating AI Summary...")
    summary_text = get_llm_summary(df)
    
    display(Markdown(f"### Trend Report: {keyword}"))
    display(Markdown(f"**Viral Duration (Past):** {past_days} days"))
    # display(Markdown(f"**Predicted Future Viral Days:** {future_days} days"))
    
    display(Markdown("#### Top 5 Trending Posts:"))
    display(Markdown(get_trending_posts_display(df)))
    
    display(Markdown("#### AI Summary:"))
    display(Markdown(summary_text))

In [53]:
# --- CELL 4: EXECUTION ---
keyword = input("Enter trend keyword to analyze: ")
if keyword:
    run_advanced_trend_demo(keyword)


Analyzing 'Russian War'... (This may take a moment)
Fetching Reddit posts...
Processing embeddings (using SentenceTransformer)... Mamba Model downstream.
Initializing Mamba Temporal Model...
Generating AI Summary...


Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Trend Report: Russian War

**Viral Duration (Past):** 114 days

#### Top 5 Trending Posts:

- [Russian tennis player Andrey Rublev writes "No war please" on the camera following his advancement t...](https://redd.it/t18ie0)
- [Ukraine: Arnold Schwarzenegger's anti-war video trends on Russian social media](https://redd.it/th8999)
- [Ukraine’s U.N. ambassador directly addressed his Russian counterpart at conclusion of U.N. Security ...](https://redd.it/t01vs2)
- [Anonymous has hacked Russian streaming services to broadcast footage of the war](https://redd.it/t8dwjp)
- [Russia’s secret documents: war in Ukraine was to last 15 days. Ukraine has seized Russian military p...](https://redd.it/t56xof)

#### AI Summary:

Russian tennis player Andrey Rublev writes "No war please" on the camera following his advancement to the final in Dubai . The Entire staff of the Russian TV channel resigned during a live stream with last words: “no war” and then played “swan lake” ballet video .